In [15]:
import numpy as np
import pandas as pd

In [16]:
# Read data
xtab = pd.read_csv("output/mcDETECT_output/xtabs/multi_marker_3D_all_5000_1000_seed_1_xtab.csv")
xtab.shape

(3137, 2933)

In [17]:
# Compute purity/completeness matrices and set up labels and thresholds

# Strip background row/col (last ones) to get detections × GT aggregates
xtab_det = xtab.iloc[:-1, :-1].copy()   # rows: spheres, cols: GTs
row_sums = xtab_det.sum(axis=1)         # total transcripts per sphere
col_sums = xtab_det.sum(axis=0)         # total transcripts per GT

# Avoid division by zero
row_sums_safe = row_sums.replace(0, np.nan)
col_sums_safe = col_sums.replace(0, np.nan)

# Purity: fraction of a sphere coming from each GT (row-normalized)
purity = xtab_det.div(row_sums_safe, axis=0)

# Completeness: fraction of a GT captured by each sphere (column-normalized)
completeness = xtab_det.div(col_sums_safe, axis=1)

# Helper labels
sphere_labels = xtab_det.index.tolist()              # e.g. ['sphere_0', ...]
gt_labels = xtab_det.columns.tolist()                # e.g. ['gt_1', ...]

# Thresholds for "good" match (can be adjusted)
TAU_P = 0.5
TAU_C = 0.5

In [18]:
# Recompute scenario breakdowns with explicit (tau_p, tau_c) logic and descriptions

# We assume xtab_det, purity, completeness, sphere_labels, gt_labels, TAU_P, TAU_C are already defined above.

# -------- GT-side: scenarios & descriptions --------

gt_records2 = []
for gt_col in gt_labels:
    counts_col = xtab_det[gt_col]
    total_gt = counts_col.sum()
    n_spheres_touch = int((counts_col > 0).sum())

    if total_gt == 0:
        scenario = "empty_gt"
        best_p = np.nan
        best_c = np.nan
    else:
        p_col = purity[gt_col]
        c_col = completeness[gt_col]
        f1_col = 2 * p_col * c_col / (p_col + c_col)
        best_idx = f1_col.idxmax()
        best_p = float(p_col.loc[best_idx]) if not pd.isna(p_col.loc[best_idx]) else 0.0
        best_c = float(c_col.loc[best_idx]) if not pd.isna(c_col.loc[best_idx]) else 0.0

        p_pass = best_p >= TAU_P
        c_pass = best_c >= TAU_C

        if n_spheres_touch == 0:
            scenario = "missed"
        elif p_pass and c_pass and n_spheres_touch == 1:
            scenario = "good_single_match"
        elif p_pass and c_pass and n_spheres_touch > 1:
            scenario = "good_multi_sphere"
        elif p_pass and not c_pass:
            scenario = "high_p_low_c"
        elif (not p_pass) and c_pass:
            scenario = "high_c_low_p"
        else:
            scenario = "low_p_low_c"

    gt_records2.append({
        "gt": gt_col,
        "total_gt": int(total_gt),
        "num_spheres_touch": n_spheres_touch,
        "best_purity": best_p,
        "best_completeness": best_c,
        "scenario": scenario,
    })


gt_summary2 = pd.DataFrame(gt_records2)

gt_scenario_descriptions = {
    "missed": "GT aggregate has no overlapping detection sphere (all transcripts in no_sphere).",
    "good_single_match": f"GT aggregate has a single detection sphere with purity >= {TAU_P} and completeness >= {TAU_C}.",
    "good_multi_sphere": f"GT aggregate has at least one good sphere (purity >= {TAU_P}, completeness >= {TAU_C}) but is also touched by multiple spheres.",
    "high_p_low_c": f"Best sphere is clean (purity >= {TAU_P}) but misses a significant fraction of GT transcripts (completeness < {TAU_C}).",
    "high_c_low_p": f"GT is mostly covered (completeness >= {TAU_C}) but the best sphere is impure (purity < {TAU_P}).",
    "low_p_low_c": "No detection sphere achieves both purity and completeness thresholds; only weak/partial coverage.",
    "empty_gt": "GT aggregate has zero assigned transcripts (degenerate case).",
}

gt_summary2["description"] = gt_summary2["scenario"].map(gt_scenario_descriptions)

print("GT scenarios (tau_p=", TAU_P, ", tau_c=", TAU_C, "):")
print(gt_summary2["scenario"].value_counts())

# -------- Sphere-side: scenarios & descriptions --------

sphere_records2 = []
for sph_label in sphere_labels:
    row = xtab_det.loc[sph_label]
    total_sphere = row.sum()
    n_gts_touch = int((row > 0).sum())

    if total_sphere == 0:
        scenario = "empty_sphere"
        best_p = np.nan
        best_c = np.nan
        frac_no_gt = np.nan
    else:
        p_row = purity.loc[sph_label]
        c_row = completeness.loc[sph_label]
        f1_row = 2 * p_row * c_row / (p_row + c_row)
        best_gt = f1_row.idxmax()
        best_p = float(p_row.loc[best_gt]) if not pd.isna(p_row.loc[best_gt]) else 0.0
        best_c = float(c_row.loc[best_gt]) if not pd.isna(c_row.loc[best_gt]) else 0.0

        mass_to_gt = row.sum()
        mass_to_no_gt = int(xtab.loc[sph_label, "no_gt"]) if "no_gt" in xtab.columns else 0
        frac_no_gt = mass_to_no_gt / (mass_to_gt + mass_to_no_gt) if (mass_to_gt + mass_to_no_gt) > 0 else 0.0

        p_pass = best_p >= TAU_P
        c_pass = best_c >= TAU_C

        if n_gts_touch == 0 and frac_no_gt > 0.9:
            scenario = "pure_background_sphere"
        elif p_pass and c_pass and n_gts_touch == 1 and frac_no_gt < 0.5:
            scenario = "good_sphere"
        elif p_pass and c_pass and (n_gts_touch > 1 or frac_no_gt >= 0.5):
            scenario = "good_but_mixed"
        elif n_gts_touch > 1 and not p_pass:
            scenario = "mixed_gt_sphere"
        elif frac_no_gt >= 0.5:
            scenario = "background_heavy_sphere"
        else:
            scenario = "low_quality_sphere"

    sphere_records2.append({
        "sphere": sph_label,
        "total_sphere": int(total_sphere),
        "num_gts_touch": n_gts_touch,
        "best_purity": best_p,
        "best_completeness": best_c,
        "frac_no_gt": frac_no_gt,
        "scenario": scenario,
    })

sphere_summary2 = pd.DataFrame(sphere_records2)

sphere_scenario_descriptions = {
    "empty_sphere": "Detection sphere contains no transcripts (degenerate case).",
    "pure_background_sphere": "Sphere overlaps no GT aggregate and >90% of its transcripts are background (no_gt).",
    "good_sphere": f"Sphere has a single dominant GT with purity >= {TAU_P} and completeness >= {TAU_C}, and <50% background.",
    "good_but_mixed": f"Sphere meets purity/completeness thresholds but also overlaps multiple GTs or has substantial background (>=50%).",
    "mixed_gt_sphere": f"Sphere overlaps multiple GT aggregates but does not reach purity >= {TAU_P} for any of them.",
    "background_heavy_sphere": "At least half of the sphere's transcripts are background (no_gt).",
    "low_quality_sphere": "Sphere has weak or ambiguous association with all GT aggregates (below thresholds).",
}

sphere_summary2["description"] = sphere_summary2["scenario"].map(sphere_scenario_descriptions)

print("\nSphere scenarios (tau_p=", TAU_P, ", tau_c=", TAU_C, "):")
print(sphere_summary2["scenario"].value_counts())

GT scenarios (tau_p= 0.5 , tau_c= 0.5 ):
scenario
good_single_match    2710
good_multi_sphere     173
empty_gt               36
high_c_low_p           12
high_p_low_c            1
Name: count, dtype: int64

Sphere scenarios (tau_p= 0.5 , tau_c= 0.5 ):
scenario
good_sphere                2873
low_quality_sphere          145
empty_sphere                 90
good_but_mixed               25
background_heavy_sphere       3
Name: count, dtype: int64
